In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import requests
import textwrap
from multiprocessing import Pool, cpu_count

c:\Users\adilo\OneDrive\Desktop\Code\FraudDetection\code\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip freeze

In [3]:
with open('data/fake.txt', 'r', encoding = 'utf-8', errors = 'ignore') as file:
    text = file.read()

In [4]:
def chunk_text(text, max_chars=600):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    paragraphs_check = []
    current = ""
    cpt = 0
    for p in paragraphs:
        p = f'Doc Number {cpt}. content: ['+ p
        p = p + ']'
        cpt += 1
        paragraphs_check.append(p)
        if len(current) + len(p) + 1 <= max_chars:
            current = (current + " " + p).strip()
        else:
            if current:
                chunks.append(current)
            current = p
    if current:
        chunks.append(current)
    return chunks, paragraphs_check

In [5]:
def encode_text(chunks):
    global model
    if model == None:
        model = SentenceTransformer('all-miniLM-L6-v2')
    
    # embed all chunks
    print('\nEncoding chunks...')
    chunk_embeddings = model.encode(chunks, batch_size= 128, show_progress_bar=False, convert_to_numpy= True, normalize_embeddings= True)
    print(f'Embeddings shape :{chunk_embeddings.shape}')
    return(chunk_embeddings)

In [16]:
print(f'encode_text: {encode_text}')

encode_text: <function encode_text at 0x0000012A84B47D80>


In [7]:
def search(query_embedding, chunks,  chunk_embedding, k = 3):
  # calculate cosine similarity   
    sim = cosine_similarity([query_embedding], chunk_embedding)
    # find similar chunks indexes
    top_idx = sim[0].argsort()[::-1][:k]

    # Find corresponding chunks to return
    top_chunks = []
    chunks_only = []
    for idx in top_idx:
        chunks_only.append(chunks[idx])
        top_chunks.append({'similarity' : sim[0][idx], 'text': chunks[idx]})
    return chunks_only, top_chunks


In [8]:
def promptLLM(system_prompt, user_prompt, query, model_name):
    
    url = 'http://localhost:11434/api/generate'
    data = {
        'model': model_name,
        'prompt': f" {system_prompt}.\nAre you ready ? Here is the context:\n{user_prompt} \n\nQuestion: {query}\n\nRemember, answer based ONLY on the context above.",
        'stream': False    
    } 

    resp = requests.post(url = url, json= data, stream= False)
    resp.raise_for_status()
    out = resp.json()
    return(out['response'])


In [9]:
def split_into_batches(lst, batch_size):
    """Yield sublists of size batch_size."""
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

In [10]:
def init_worker():
    global model
    if model is None:
        print('loading model in worker...')
        model = SentenceTransformer('all-miniLM-L6-v2')
        print('model loaded in worker...')

In [11]:
def multiprocess_encoding(chunks):
    # number of cores to use
    n_proc = max(1, cpu_count()-1)
    print(f'Cores To use: {n_proc}')

    # Devide chunks into batches each to be precessed in a separate core
    print('affecting chunks ...')
    chunk_splits = np.array_split(chunks, n_proc)
    
    # Launch multiprocessing
    print('Multiprocessing ...')
    with Pool(processes= n_proc, initializer= init_worker) as pool:
        results =pool.map(encode_text, chunk_splits)

    print('Merging back results ...')
    return np.vstack(results)

In [12]:
a = [1,2,3]
b= [4,5,6]
np.vstack((a,b))

array([[1, 2, 3],
       [4, 5, 6]])

In [13]:
model = None

In [14]:
def customRAG(context, query, best_chunks= None):

   # chunking text
    chunks, docs_check = chunk_text(context)

    # embedding query and chunks into vectors (384 dimension)
    query_embedding = encode_text(query)
    chunks_embeddings = multiprocess_encoding(chunks) 
    print(f'Quick check: chunks: {len(chunks_embeddings)}, one chunk embedding: {len(chunks_embeddings(0))} ')
    
    # retrieving top (k) similar chunks to query
    if best_chunks:
        retrieved_chunks = best_chunks
    else:
        retrieved_chunks, _ = search(query_embedding, chunks, chunks_embeddings, k= 5)

    # Prompt Preparation
    system_prompt=( "You are an assistant that answers questions based ONLY on the provided context. "
        "Give clearest full answer based on the context only, and quote from context to back up your answer"
        "BUT, If the context does not contain the answer, say it is not mentionned. "
        "Do not invent information."
        "At the end, make a separation from previous text, and answer from your own knowledge even out of provided context. \n\n")
        
    context_text = "\n\n".join(retrieved_chunks)
    user_prompt= f"Context:\n{context_text}\n\nQuestion: What did Trump say about immigrants ?\n\nAnswer based only on the context above."
    model_name = 'llama3'

    # sending POST Ollama server on localhost sepecifying model and prompt
    answer = promptLLM(system_prompt, user_prompt, query, model_name)
    return answer

In [15]:
""" query = 'What Did Donald Trump say regarding immigrants ?'
customRAG(text, query) """

" query = 'What Did Donald Trump say regarding immigrants ?'\ncustomRAG(text, query) "